# 05 — CLIP Clustering V2 : UMAP + HDBSCAN

# V1 (PCA 50D + KMeans k=5) abandonnée — clusters fourre-tout, doublons de labels, k fixé arbitrairement.
#
# V2 : UMAP (réduction non-linéaire, métrique cosine) + HDBSCAN (clustering densité, k automatique, outliers = -1).
#
# Input  : data/warehouse/photo_embeddings/
# Output : data/warehouse/photo_clusters/

In [19]:
import sys, os
from pathlib import Path

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import WAREHOUSE

EMBEDDINGS_DIR = Path(WAREHOUSE) / 'photo_embeddings'
CLUSTERS_DIR   = Path(WAREHOUSE) / 'photo_clusters'

CLIP_MODEL = 'openai/clip-vit-large-patch14'

CANDIDATE_LABELS = [
      'soiree entre amis',
      'concert ou festival',
      'voyage et paysage',
      'nourriture et restaurant',
      'sport et fitness',
      'famille',
      'ville et architecture',
      'nature et randonnee',
      'selfie portrait',
      'plage et vacances',
      'meme et humour internet',
      'photo du quotidien entre amis',
      'photo de voiture',
]

print(f'Embeddings : {EMBEDDINGS_DIR}')
print(f'Clusters   : {CLUSTERS_DIR}')

Embeddings : /opt/spark/data/warehouse/photo_embeddings
Clusters   : /opt/spark/data/warehouse/photo_clusters


## 1. Lecture des embeddings via Spark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('MyDigitalTwin-CLIP-Clustering-V2')
    .master('local[*]')
    # 4g suffisant : les embeddings (2419 × 768 floats ≈ 7 Mo) sont collectés en numpy.
    # UMAP et HDBSCAN tournent entièrement sur le driver — Spark sert uniquement
    # à lire le Parquet et à sauvegarder le résultat final.
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

df = spark.read.parquet(str(EMBEDDINGS_DIR)).cache()
print(f'Photos chargées : {df.count()}')
df.printSchema()

## 2. Collect -> numpy

Pour 2 419 photos, un seul `.collect()` est acceptable. UMAP et HDBSCAN ne sont pas dans MLlib.

In [21]:
import numpy as np

print('Collecte des embeddings...')
rows = df.select('path', 'filename', 'embedding').collect()

paths     = [r['path']     for r in rows]
filenames = [r['filename'] for r in rows]
X = np.array([r['embedding'] for r in rows], dtype=np.float32)

print(f'Shape embeddings : {X.shape}')
print(f'Norme L2 moyenne : {np.linalg.norm(X, axis=1).mean():.4f}  (attendu ~1.0)')

Collecte des embeddings...


Shape embeddings : (2419, 768)
Norme L2 moyenne : 1.0000  (attendu ~1.0)


## 3. UMAP — 768D -> 50D (clustering) + 2D (visualisation)

In [ ]:
import umap

print('1/2 - UMAP 768D -> 50D (clustering)...')
reducer_50 = umap.UMAP(
    # n_components : dimension de l'espace réduit pour HDBSCAN.
    # 768D → 50D : réduit le bruit dimensionnel sans écraser la structure locale.
    # Trop bas (ex : 2D) = perte d'information pour le clustering.
    # Trop haut (ex : 200D) = malédiction de la dimensionnalité pour HDBSCAN.
    n_components=50,

    # n_neighbors : taille du voisinage local pour construire le graphe de similarité.
    # Petit (ex : 5) = structure fine et locale, sensible au bruit.
    # Grand (ex : 50) = structure globale, clusters plus larges.
    # 15 est le défaut UMAP — bon équilibre sur ~2400 points.
    n_neighbors=15,

    # min_dist : distance minimale entre deux points dans l'espace réduit.
    # 0.0 = points compressés en clusters très denses.
    # 1.0 = distribution quasi-uniforme, structure locale perdue.
    # 0.1 : préserve la structure locale sans sur-comprimer.
    min_dist=0.1,

    # metric : distance utilisée dans l'espace original (768D).
    # Embeddings CLIP L2-normalisés → cosine distance = distance euclidienne sur l'hypersphère.
    # Utiliser 'cosine' explicitement est plus correct que 'euclidean' ici.
    metric='cosine',

    random_state=42,
    verbose=True,
)
X_50 = reducer_50.fit_transform(X)
print(f'Shape après UMAP 50D : {X_50.shape}')

print('2/2 - UMAP 768D -> 2D (visualisation dashboard)...')
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
)
X_2d = reducer_2d.fit_transform(X)
print(f'Shape UMAP 2D : {X_2d.shape}')

## 4. HDBSCAN — clustering densite (k automatique)

In [ ]:
import hdbscan
import pandas as pd

print('HDBSCAN...')
clustering = hdbscan.HDBSCAN(
    # min_cluster_size : taille minimale d'un cluster.
    # Sur 2419 photos, 50 ≈ 2% du corpus.
    # Trop petit (ex : 10) → micro-clusters parasites (testé avec 30 : trop fragmenté).
    # Trop grand (ex : 200) → clusters fusionnés, perte de granularité.
    min_cluster_size=50,

    # min_samples : nombre de voisins requis pour qu'un point soit "core point".
    # Petit (ex : 1) = moins de bruit, clusters plus inclusifs.
    # Grand (ex : 20) = clusters plus denses, plus de points classés bruit (-1).
    # 5 : bruit modéré — les photos atypiques vont en bruit sans forcer leur assignation.
    min_samples=5,

    # metric : distance utilisée sur l'espace UMAP 50D.
    # Ici 'euclidean' est correct : UMAP produit des coordonnées cartésiennes,
    # pas des embeddings normalisés. Différent de la métrique cosine utilisée dans UMAP.
    metric='euclidean',

    # cluster_selection_method : stratégie de sélection des clusters dans la hiérarchie.
    # 'eom' (Excess of Mass) : favorise les clusters stables de tailles inégales — réaliste
    # ici (soirées = 1295 photos vs enfance = 67 photos).
    # Alternatif 'leaf' : clusters plus petits et équilibrés, moins adapté à ce corpus.
    cluster_selection_method='eom',
)
labels = clustering.fit_predict(X_50)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()

print(f'Clusters trouvés : {n_clusters}')
print(f'Outliers (bruit) : {n_noise} photos ({n_noise/len(labels)*100:.1f}%)')

print('\nDistribution :')
series = pd.Series(labels)
print(series.value_counts().sort_index().rename(index={-1: 'bruit (-1)'}).to_string())

## 5. Labels manuels (basés sur inspection visuelle des clusters)

In [24]:
cluster_labels = {
  -1: "photos diverses",
   0: "photos d'enfance",
   1: "amis en voyage",
   2: "memes et humour",
   3: "quotidien entre amis",
   4: "photos diverses",   # fourre-tout, honnete
   5: "soirees",
}

print("Labels :")
for cid, label in sorted(cluster_labels.items()):
  mask = labels == cid
  print(f"  Cluster {cid:>2} ({mask.sum():>4} photos) -> {label}")

Labels :
  Cluster -1 ( 323 photos) -> photos diverses
  Cluster  0 (  67 photos) -> photos d'enfance
  Cluster  1 (  74 photos) -> amis en voyage
  Cluster  2 (  81 photos) -> memes et humour
  Cluster  3 ( 253 photos) -> quotidien entre amis
  Cluster  4 ( 326 photos) -> photos diverses
  Cluster  5 (1295 photos) -> soirees


## 6. Sauvegarde via Spark

In [25]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

rows_out = [
    (
        paths[i],
        filenames[i],
        int(labels[i]),
        cluster_labels[int(labels[i])],
        float(X_2d[i, 0]),
        float(X_2d[i, 1]),
    )
    for i in range(len(paths))
]

schema = StructType([
    StructField('path',          StringType(),  nullable=False),
    StructField('filename',      StringType(),  nullable=False),
    StructField('cluster',       IntegerType(), nullable=False),
    StructField('cluster_label', StringType(),  nullable=False),
    StructField('umap_x',         FloatType(),   nullable=False),
    StructField('umap_y',         FloatType(),   nullable=False),
])

df_out = spark.createDataFrame(rows_out, schema=schema)

CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
df_out.write.mode('overwrite').parquet(str(CLUSTERS_DIR))

print(f'Clusters sauvegardes -> {CLUSTERS_DIR}')
spark.read.parquet(str(CLUSTERS_DIR)).groupBy('cluster', 'cluster_label').count().orderBy('cluster').show(truncate=False)

Clusters sauvegardes -> /opt/spark/data/warehouse/photo_clusters
+-------+--------------------+-----+
|cluster|cluster_label       |count|
+-------+--------------------+-----+
|-1     |photos diverses     |323  |
|0      |photos d'enfance    |67   |
|1      |amis en voyage      |74   |
|2      |memes et humour     |81   |
|3      |quotidien entre amis|253  |
|4      |photos diverses     |326  |
|5      |soirees             |1295 |
+-------+--------------------+-----+



In [26]:
spark.stop()